# Data cleaning: removing unnecessary columns and rows

**Input:** `data/submissions_trump.parquet` and `data/comments_trump.parquet`, produced by `scripts/extract_direct_trump_mentions.py` and `scripts/combine_datasets.py`.
**Output:** `data/cleaned/` holds the cleaned submissions and comments as Parquet (for analysis) and CSV (the format the assignment requires).

The steps below follow the data profiling done earlier:

1. **Columns.** Keep only the fields that matter for studying attitudes toward Trump over time. Drop empty, constant, duplicated and purely presentational API fields.
2. **Submission rows.**
   - Replace the `[deleted]` / `[removed]` placeholders in `selftext` with an empty string, and record that in an `is_removed` flag. Rows stay, because the title is still genuine user text.
   - Drop the few posts whose *title* is also junk.
   - Submissions that do not mention Trump themselves (`submission_mentions_trump == False`) are **left untouched** for now.
3. **Comment rows.**
   - Remove bots and moderator messages.
   - Remove exact duplicates (same author and same text).
   - Build a `body_clean` text with quoted replies, URLs and HTML entities stripped.
   - Flag (not drop) comments that mention Trump only inside a quoted reply.
4. **Normalization.** Add datetime and period columns, and harmonize the r/AskTrumpSupporters flairs into supporter / nonsupporter / undecided.

The comments file is large (~4.6M rows, 1.3 GB compressed, ~7 GB in RAM as a DataFrame), so comments are processed **in streaming batches** and written incrementally. The notebook never holds the whole comments table in memory.

In [1]:
import html
import re
import time
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA = Path.cwd().parent / "data"

SRC_DIR = DATA / "processed" / "combined"

SRC_SUBMISSIONS = SRC_DIR / "submissions_trump.parquet"
SRC_COMMENTS = SRC_DIR / "comments_trump.parquet"

OUT = DATA / "cleaned"
OUT.mkdir(parents=True, exist_ok=True)

SAVE_CSV = False        # the assignment asks for CSV; Parquet is always written too
BATCH_SIZE = 200_000     # comment rows per streaming batch; lower it if RAM is tight

print("data dir:", DATA)

data dir: c:\PythonProjects\reddit_analysis_project\data


## 1. Column selection

The raw files carry **132 submission columns** and **87 comment columns**, most of them Reddit API noise. A column was dropped if it is:

| Reason | Examples |
|---|---|
| Completely empty (all null) | `view_count`, `approved_by`, `banned_by`, `likes`, `mod_note`, `removed_by`, `location_*`, `num_reports` |
| Constant (one value only) | `archived`, `hidden`, `clicked`, `saved`, `visited`, `downs`, `awarders`, `gildings`, `treatment_tags` |
| Duplicate of another column | `subreddit_id`, `subreddit_name_prefixed`, `name` (= `t3_` + `id`), `created` (= `created_utc`), `body_html`, `selftext_html`, `permalink`, `link_id` (= `submission_id`) |
| Media / styling only | `media*`, `secure_media*`, `preview`, `thumbnail*`, `gallery_data`, `*_css_class`, `*_color`, `*_richtext`, `*_template_id`, `websocket_url`, `_meta` |
| Sparse and irrelevant | `crosspost_*`, `poll_data`, `author_premium`, `profile_img`, `link_*` (0.01% filled) |

What remains covers **who** (author, flair), **where** (subreddit), **when** (timestamp), **what** (text) and **how it was received** (score, comments, controversiality, upvote ratio). It also keeps the IDs needed to link comments to posts and replies to parents.

In [2]:
SUBMISSION_COLUMNS = [
    "id", "created_utc", "source_month", "subreddit", "author", "author_flair_text",
    "title", "selftext", "is_self", "domain", "url", "link_flair_text",
    "score", "num_comments", "upvote_ratio",   # upvote_ratio exists only for 2025-2026
    "removed_by_category",                      # exists only for 2025-2026
    "distinguished", "stickied",
    "submission_mentions_trump", "has_trump_comment",
]

COMMENT_COLUMNS = [
    "id", "submission_id", "parent_id", "created_utc", "source_month",
    "subreddit", "author", "author_flair_text", "body",
    "score", "controversiality", "is_submitter", "distinguished", "stickied", "edited",
]

for name, path, keep in [("submissions", SRC_SUBMISSIONS, SUBMISSION_COLUMNS),
                         ("comments", SRC_COMMENTS, COMMENT_COLUMNS)]:
    all_cols = pq.ParquetFile(path).schema_arrow.names
    missing = set(keep) - set(all_cols)
    assert not missing, f"{name}: missing columns {missing}"
    print(f"{name:12s} {len(all_cols):>4} columns -> keep {len(keep):>3}, drop {len(all_cols) - len(keep)}")

submissions   132 columns -> keep  20, drop 112
comments       87 columns -> keep  15, drop 72


## 2. Shared helpers

**Period labels.** The data covers four six-month windows: January–June of 2017, 2018, 2025 and 2026. These map onto the first and second year of each Trump term, so the two terms can be compared at the same point in the presidency.

**Flair normalization (r/AskTrumpSupporters).** In this subreddit the author flair is a self-declared stance. The flair names changed over the years ("Nimble Navigator" → "Trump Supporter", "Non-Trump Supporter" → "Nonsupporter", plus trailing spaces), so they are mapped to one `stance` column. "Toaster" appears only in 2018 and its meaning is unclear, so it becomes `other`.

In [3]:
PERIOD = {"2017": "T1_Y1", "2018": "T1_Y2", "2025": "T2_Y1", "2026": "T2_Y2"}

STANCE_MAP = {
    "nimble navigator": "supporter",
    "trump supporter": "supporter",
    "non-trump supporter": "nonsupporter",
    "nonsupporter": "nonsupporter",
    "undecided": "undecided",
}


def add_time_columns(df: pd.DataFrame) -> pd.DataFrame:
    df["created_at"] = pd.to_datetime(df["created_utc"], unit="s", utc=True)
    df["year"] = df["source_month"].str[:4].astype("int16")
    df["period"] = df["source_month"].str[:4].map(PERIOD)
    df["term"] = df["period"].str[:2]            # T1 = 2017-2018, T2 = 2025-2026
    return df


def normalize_flair(df: pd.DataFrame) -> pd.DataFrame:
    flair = df["author_flair_text"].astype("string").str.strip().replace("", pd.NA)
    df["author_flair_text"] = flair
    is_ats = df["subreddit"].eq("AskTrumpSupporters")
    stance = flair.str.lower().map(STANCE_MAP)
    stance = stance.where(stance.notna() | flair.isna(), "other")
    df["stance"] = stance.where(is_ats).astype("string")   # only meaningful inside r/AskTrumpSupporters
    return df

## 3. Submissions

The submissions table (~0.5M rows) fits in memory, so it is loaded in one go with only the selected columns.

In [4]:
subs = pd.read_parquet(SRC_SUBMISSIONS, columns=SUBMISSION_COLUMNS)
n_subs_raw = len(subs)
print(f"raw submissions: {n_subs_raw:,}")
subs.head(3)

raw submissions: 513,848


,id,created_utc,source_month,subreddit,author,author_flair_text,title,selftext,is_self,domain,url,link_flair_text,score,num_comments,upvote_ratio,removed_by_category,distinguished,stickied,submission_mentions_trump,has_trump_comment
0,5lch1i,1483228990,2017-01,politics,Gaybro1992,NaN,Trump ditches press pool to play golf,NaN,False,cnn.com,http://www.cnn.com/2016/12/31/politics/donald-...,Off-Topic,1,4,NaN,NaN,NaN,False,True,False
1,5lci8r,1483229424,2017-01,politics,gazil9,NaN,"Putin won 2016, but Russia has its limits as a...",NaN,False,washingtonpost.com,https://www.washingtonpost.com/world/europe/pu...,NaN,266,49,NaN,NaN,NaN,False,False,True
2,5lciy3,1483229651,2017-01,politics,[deleted],NaN,This Photo of a Trump Billboard in Mumbai is R...,[deleted],False,m.huffpost.com,http://m.huffpost.com/us/entry/us_586813c3e4b0...,No Link Shorteners,1,1,NaN,NaN,NaN,False,True,False


### 3.1 Placeholder text in `selftext`

A deleted post has its body replaced by `[deleted]`, and a post removed by moderators has it replaced by `[removed]`. Reddit never blanks the **title**, so these posts still hold real user text in their headline.

- We **keep the rows** and set `selftext` to an empty string. A literal `[removed]` would otherwise be read as a word by sentiment and topic models.
- The flag `is_removed` keeps the information. Removal rates by subreddit and year are a useful moderation signal in their own right.

In [5]:
PLACEHOLDERS = ["[deleted]", "[removed]"]

subs["is_removed"] = subs["selftext"].isin(PLACEHOLDERS)
print(subs["selftext"].where(subs["is_removed"]).value_counts())
print(f"share of all submissions with placeholder body: {subs['is_removed'].mean():.1%}")
print(f"share among text posts (is_self): {subs.loc[subs['is_self'], 'is_removed'].mean():.1%}")

subs["selftext"] = subs["selftext"].where(~subs["is_removed"], "").fillna("")
subs["author_deleted"] = subs["author"].eq("[deleted]")

selftext
[deleted]    67985
[removed]    41029
Name: count, dtype: int64
share of all submissions with placeholder body: 21.2%
share among text posts (is_self): 71.9%


### 3.2 Junk titles

A small group of posts has no usable text even in the title:

- `[ Removed by moderator ]`
- `?I_do_not_support_Trump` / `?I_support_Trump` and their variants. These are removed r/AskTrumpSupporters posts whose title was overwritten.

These rows are dropped.

In [6]:
JUNK_TITLE = re.compile(r"^\W*(?:\[\s*removed by moderator\s*\]|I_(?:do_not_)?support_trump)\W*$", re.IGNORECASE)

junk_title = subs["title"].fillna("").str.match(JUNK_TITLE)
print(subs.loc[junk_title, "title"].value_counts().head(10))
print(f"\njunk-title rows to drop: {junk_title.sum():,}")

subs = subs.loc[~junk_title].copy()

title
?I_do_not_support_Trump      4944
?I_support_Trump             1293
[ Removed by moderator ]      261
I_do_not_support_Trump         49
?I_do_not_support_trump        15
?I_Support_Trump               14
I_do_not_support_trump         14
I_support_Trump                 6
I_support_trump                 6
"?I_do_not_support_Trump"       5
Name: count, dtype: int64

junk-title rows to drop: 6,642


### 3.3 Duplicates, normalization, combined text

- `id` is already unique, and the `assert` checks this.
- `text` = title + body. This is the single field to feed into text models.
- Moderator / bot submissions (for example AutoModerator megathreads) are **not dropped**, because many user comments hang under them. They are flagged in `is_mod_post` so text analyses can exclude them.

In [7]:
assert subs["id"].is_unique

BOT_AUTHORS = {
    "AutoModerator", "PoliticsModeratorBot", "autotldr", "AskTrumpSupporters-ModTeam",
    "sneakpeekbot", "tweettranscriberbot", "TotesMessenger", "FatFingerHelperBot",
    "video_descriptionbot", "WikiTextBot", "youtubefactsbot", "Mentioned_Videos", "SmallSubBot",
}

subs["is_mod_post"] = subs["distinguished"].eq("moderator") | subs["author"].isin(BOT_AUTHORS)
subs["text"] = (subs["title"].fillna("") + "\n\n" + subs["selftext"]).str.strip()

subs = add_time_columns(subs)
subs = normalize_flair(subs)

subs = subs.astype({"score": "int32", "num_comments": "int32"})
subs = subs.sort_values("created_utc").reset_index(drop=True)

print(f"submissions: {n_subs_raw:,} -> {len(subs):,} (dropped {n_subs_raw - len(subs):,})")
print("mod/bot posts flagged:", int(subs["is_mod_post"].sum()))
print("\nsubmission_mentions_trump (left untouched):")
print(subs["submission_mentions_trump"].value_counts())
subs.head(3)

submissions: 513,848 -> 507,206 (dropped 6,642)
mod/bot posts flagged: 578

submission_mentions_trump (left untouched):
submission_mentions_trump
True     343974
False    163232
Name: count, dtype: Int64


,id,created_utc,source_month,subreddit,author,author_flair_text,title,selftext,is_self,domain,url,link_flair_text,score,num_comments,upvote_ratio,removed_by_category,distinguished,stickied,submission_mentions_trump,has_trump_comment,is_removed,author_deleted,is_mod_post,text,created_at,year,period,term,stance
0,5lch1i,1483228990,2017-01,politics,Gaybro1992,<NA>,Trump ditches press pool to play golf,,False,cnn.com,http://www.cnn.com/2016/12/31/politics/donald-...,Off-Topic,1,4,NaN,NaN,NaN,False,True,False,False,False,False,Trump ditches press pool to play golf,2017-01-01 00:03:10+00:00,2017,T1_Y1,T1,<NA>
1,5lci8r,1483229424,2017-01,politics,gazil9,<NA>,"Putin won 2016, but Russia has its limits as a...",,False,washingtonpost.com,https://www.washingtonpost.com/world/europe/pu...,NaN,266,49,NaN,NaN,NaN,False,False,True,False,False,False,"Putin won 2016, but Russia has its limits as a...",2017-01-01 00:10:24+00:00,2017,T1_Y1,T1,<NA>
2,5lciy3,1483229651,2017-01,politics,[deleted],<NA>,This Photo of a Trump Billboard in Mumbai is R...,,False,m.huffpost.com,http://m.huffpost.com/us/entry/us_586813c3e4b0...,No Link Shorteners,1,1,NaN,NaN,NaN,False,True,False,True,True,False,This Photo of a Trump Billboard in Mumbai is R...,2017-01-01 00:14:11+00:00,2017,T1_Y1,T1,<NA>


## 4. Comments

Comments are cleaned batch by batch. Each batch goes through these steps:

1. **Bots and moderator messages are removed.** These are `distinguished == "moderator"` plus a curated list of real bots (AutoModerator, PoliticsModeratorBot, autotldr, …).
   A naive `"bot" in author` filter is **not** used. Checking samples showed that accounts like `granola_robot` or `BadAdviceBot` are ordinary users writing normal political comments.
2. **Exact duplicates are removed**: the same author posting the same text (spam, copy-paste, double submits). The check spans all batches and uses a set of 64-bit hashes. `[deleted]` authors are exempt, because different people share that name.
3. **`body_clean` is built:**
   - quoted lines (`> ...`, stored as `&gt; ...`) are removed, since they are someone else's words;
   - markdown links `[text](url)` become `text`, and bare URLs are removed;
   - HTML entities (`&amp;`, `&lt;`, zero-width spaces) are unescaped;
   - whitespace is collapsed.
4. **The Trump mention is re-checked on `body_clean`.** The original filter was `\btrump\b` on the raw body, so it also matched comments where "Trump" appears **only inside a quote**, i.e. in someone else's words. Example:
   ```
   > Trump has done more for the economy than any president
   Source?
   ```
   The author's own text here is just "Source?". Scoring the quote would attribute someone else's opinion to the author and count the same text twice. These comments are **kept but flagged** with `trump_only_in_quote = True`. Each analysis can then decide whether to include them: exclude them for sentiment toward Trump, include them for discussion volume or reply networks.
   Comments that are *empty* after cleaning (only a quote and/or a link, no own text) are dropped, since they contain no words by the author.
5. **Very short comments are kept.** "Fuck Trump" or "Thanks Trump!" are among the most frequent comments and are a clear attitude signal.

Rows whose author is `[deleted]` are kept, because their text is intact. The `author_deleted` flag lets author-level analyses exclude them.

In [8]:
TRUMP = re.compile(r"\btrump\b", re.IGNORECASE)
QUOTE_LINE = r"(?m)^[ \t]*(?:&gt;|>).*$"
MD_LINK = r"\[([^\]]*)\]\((?:https?://|/)[^)]*\)"
URL = r"(?:https?://|www\.)\S+"


def clean_body(body: pd.Series) -> pd.Series:
    s = body.fillna("").astype(str)
    s = s.str.replace(QUOTE_LINE, "", regex=True)
    s = s.str.replace(MD_LINK, r"\1", regex=True)
    s = s.str.replace(URL, "", regex=True)
    s = s.map(html.unescape)
    s = s.str.replace("​", "", regex=False)
    s = s.str.replace(r"[ \t]+", " ", regex=True)
    s = s.str.replace(r"\s*\n\s*", "\n", regex=True).str.strip()
    return s


# quick sanity check on a toy example
demo = pd.Series(["&gt; Trump said X\n\nI disagree with that &amp; so does [this](https://x.com/a).",
                  "&gt; quoted text\nTrump is doing fine https://t.co/abc"])
pd.DataFrame({"raw": demo, "clean": clean_body(demo), "still_mentions_trump": clean_body(demo).str.contains(TRUMP)})

,raw,clean,still_mentions_trump
0,&gt; Trump said X\n\nI disagree with that &amp...,I disagree with that & so does this.,False
1,&gt; quoted text\nTrump is doing fine https://...,Trump is doing fine,True


In [9]:
OUT_COMMENTS_PQ = OUT / "comments_clean.parquet"
OUT_COMMENTS_CSV = OUT / "comments_clean.csv"

stats = {"raw": 0, "bot_or_mod": 0, "duplicate": 0, "empty_after_clean": 0, "kept": 0, "kept_trump_only_in_quote": 0}
by_bot = pd.Series(dtype="int64")
seen_hashes: set[int] = set()

writer = None
first_csv = True
t0 = time.time()

src = pq.ParquetFile(SRC_COMMENTS)
for i, batch in enumerate(src.iter_batches(batch_size=BATCH_SIZE, columns=COMMENT_COLUMNS), start=1):
    df = batch.to_pandas()
    stats["raw"] += len(df)

    # 1. bots / moderator messages
    bot = df["distinguished"].eq("moderator") | df["author"].isin(BOT_AUTHORS)
    stats["bot_or_mod"] += int(bot.sum())
    by_bot = by_bot.add(df.loc[bot, "author"].value_counts(), fill_value=0)
    df = df.loc[~bot]

    # 2. exact duplicates (author + body), across all batches
    h = pd.util.hash_pandas_object(df[["author", "body"]], index=False).to_numpy()
    real_author = df["author"].ne("[deleted]").to_numpy()
    dup = pd.Series(h).duplicated().to_numpy() | pd.Series(h).isin(seen_hashes).to_numpy()
    dup &= real_author
    seen_hashes.update(h[real_author].tolist())
    stats["duplicate"] += int(dup.sum())
    df = df.loc[~dup].copy()

    # 3. clean text
    df["body_clean"] = clean_body(df["body"])
    empty = df["body_clean"].eq("")
    stats["empty_after_clean"] += int(empty.sum())
    df = df.loc[~empty].copy()
    df["trump_only_in_quote"] = ~df["body_clean"].str.contains(TRUMP)
    stats["kept_trump_only_in_quote"] += int(df["trump_only_in_quote"].sum())

    # 4. normalization
    df["author_deleted"] = df["author"].eq("[deleted]")
    df["is_top_level"] = df["parent_id"].str.startswith("t3_")
    df["edited"] = df["edited"].astype("string").ne("False")
    df = add_time_columns(df)
    df = normalize_flair(df)
    df = df.drop(columns=["distinguished"])     # all remaining rows are non-moderator now
    df = df.astype({"score": "int32", "controversiality": "int8", "is_submitter": "boolean"})
    # fixed dtypes so every batch has the same Parquet schema (a batch with an all-null column would otherwise differ)
    str_cols = ["id", "submission_id", "parent_id", "source_month", "subreddit", "author", "body", "body_clean", "period", "term"]
    df[str_cols] = df[str_cols].astype("string")

    stats["kept"] += len(df)

    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUT_COMMENTS_PQ, table.schema, compression="snappy")
    writer.write_table(table)

    if SAVE_CSV:
        df.to_csv(OUT_COMMENTS_CSV, mode="w" if first_csv else "a", header=first_csv, index=False, encoding="utf-8")
        first_csv = False

    print(f"batch {i:>3} | read {stats['raw']:>9,} | kept {stats['kept']:>9,} | {time.time() - t0:6.0f}s")

writer.close()
del seen_hashes

batch   1 | read   200,000 | kept   196,547 |     10s
batch   2 | read   400,000 | kept   391,912 |     30s
batch   3 | read   600,000 | kept   587,458 |     48s
batch   4 | read   800,000 | kept   783,192 |     67s
batch   5 | read 1,000,000 | kept   979,008 |     87s
batch   6 | read 1,200,000 | kept 1,174,553 |    109s
batch   7 | read 1,400,000 | kept 1,370,071 |    128s
batch   8 | read 1,600,000 | kept 1,565,928 |    146s
batch   9 | read 1,800,000 | kept 1,761,733 |    175s
batch  10 | read 2,000,000 | kept 1,957,051 |    197s
batch  11 | read 2,200,000 | kept 2,152,371 |    210s
batch  12 | read 2,400,000 | kept 2,347,372 |    220s
batch  13 | read 2,600,000 | kept 2,542,525 |    231s
batch  14 | read 2,800,000 | kept 2,737,676 |    243s
batch  15 | read 3,000,000 | kept 2,932,711 |    259s
batch  16 | read 3,200,000 | kept 3,128,064 |    276s
batch  17 | read 3,400,000 | kept 3,323,404 |    288s
batch  18 | read 3,600,000 | kept 3,518,422 |    302s
batch  19 | read 3,800,000 |

In [10]:
comment_summary = pd.Series(stats, name="rows").to_frame()
comment_summary["% of raw"] = (100 * comment_summary["rows"] / stats["raw"]).round(2)
# note: kept_trump_only_in_quote is a subset of kept (flagged, not removed)
print("Removed bot / moderator accounts:")
print(by_bot.sort_values(ascending=False).astype(int).head(15).to_string())
comment_summary

Removed bot / moderator accounts:
author
AutoModerator                 35151
PoliticsModeratorBot           5749
autotldr                       5118
AskTrumpSupporters-ModTeam      343
fullstep                        222
touchmystuffIkillyou            120
likeafox                        102
wenchette                        85
sneakpeekbot                     40
Asukan                           38
cuddlefishcat                    29
bluemexico                       27
therealdanhill                   24
mod1fier                         24
Anxa                             23


,rows,% of raw
raw,4601620,100.00
bot_or_mod,47417,1.03
duplicate,27090,0.59
empty_after_clean,27599,0.60
kept,4499514,97.78
kept_trump_only_in_quote,248178,5.39


## 5. Save submissions

In [11]:
OUT_SUBS_PQ = OUT / "submissions_clean.parquet"
OUT_SUBS_CSV = OUT / "submissions_clean.csv"

subs.to_parquet(OUT_SUBS_PQ, index=False)
if SAVE_CSV:
    subs.to_csv(OUT_SUBS_CSV, index=False, encoding="utf-8")
print("saved", OUT_SUBS_PQ.name, "and", OUT_SUBS_CSV.name if SAVE_CSV else "(no csv)")

saved submissions_clean.parquet and (no csv)


## 6. Result overview

Here are the before/after row counts, file sizes and column counts. The assignment requires the filtered dataset to be **≥ 1 GB**; the CSV files are the relevant measure for that.

In [12]:
def mb(p: Path) -> float:
    return p.stat().st_size / 1024**2 if p.exists() else float("nan")

cm = pq.ParquetFile(OUT_COMMENTS_PQ).metadata
overview = pd.DataFrame({
    "rows_before": [n_subs_raw, stats["raw"]],
    "rows_after": [len(subs), cm.num_rows],
    "cols_before": [pq.ParquetFile(SRC_SUBMISSIONS).metadata.num_columns, pq.ParquetFile(SRC_COMMENTS).metadata.num_columns],
    "cols_after": [subs.shape[1], cm.num_columns],
    "parquet_MB": [mb(OUT_SUBS_PQ), mb(OUT_COMMENTS_PQ)],
    "csv_MB": [mb(OUT_SUBS_CSV), mb(OUT_COMMENTS_CSV)],
}, index=["submissions", "comments"])
overview["rows_removed_%"] = (100 * (1 - overview["rows_after"] / overview["rows_before"])).round(2)
overview.loc["TOTAL"] = overview.sum(numeric_only=True)
overview.loc["TOTAL", "rows_removed_%"] = round(100 * (1 - overview.loc["TOTAL", "rows_after"] / overview.loc["TOTAL", "rows_before"]), 2)
overview.round(1)

,rows_before,rows_after,cols_before,cols_after,parquet_MB,csv_MB,rows_removed_%
submissions,513848.0,507206.0,132.0,29.0,98.7,NaN,1.3
comments,4601620.0,4499514.0,87.0,23.0,2032.3,NaN,2.2
TOTAL,5115468.0,5006720.0,219.0,52.0,2131.0,0.0,2.1


In [13]:
# Comments per period x subreddit after cleaning (streamed, two small columns)
cc = pd.read_parquet(OUT_COMMENTS_PQ, columns=["period", "subreddit"])
pd.crosstab(cc["subreddit"], cc["period"], margins=True)

period,T1_Y1,T1_Y2,T2_Y1,T2_Y2,All
subreddit,,,,,
AskTrumpSupporters,69370,41507,19895,19246,150018
Conservative,26661,22751,40642,12001,102055
PoliticalDiscussion,39489,17307,25552,10924,93272
Republican,6566,1099,6776,1141,15582
democrats,3226,5089,34299,11084,53698
politics,1573106,1275335,722928,513520,4084889
All,1718418,1363088,850092,567916,4499514


In [14]:
# Normalized stance in r/AskTrumpSupporters comments, by period
cs = pd.read_parquet(OUT_COMMENTS_PQ, columns=["period", "stance", "subreddit"],
                     filters=[("subreddit", "==", "AskTrumpSupporters")])
pd.crosstab(cs["stance"].fillna("no flair"), cs["period"], margins=True)

period,T1_Y1,T1_Y2,T2_Y1,T2_Y2,All
stance,,,,,
no flair,4773,525,42,63,5403
nonsupporter,35883,25354,11018,11375,83630
other,1299,11,3,3,1316
supporter,23585,14035,8332,7322,53274
undecided,3830,1582,500,483,6395
All,69370,41507,19895,19246,150018


In [15]:
# Share of comments where Trump appears only inside a quote, by period
cq = pd.read_parquet(OUT_COMMENTS_PQ, columns=["period", "trump_only_in_quote"])
cq.groupby("period")["trump_only_in_quote"].agg(["sum", "mean"]).rename(columns={"sum": "flagged", "mean": "share"}).round(3)

,flagged,share
period,,
T1_Y1,104682,0.061
T1_Y2,86255,0.063
T2_Y1,35780,0.042
T2_Y2,21461,0.038


In [16]:
# Peek at the cleaned comments
pq.ParquetFile(OUT_COMMENTS_PQ).read_row_group(0).slice(0, 5).to_pandas()[
    ["created_at", "subreddit", "author", "stance", "score", "trump_only_in_quote", "body", "body_clean"]
]

,created_at,subreddit,author,stance,score,trump_only_in_quote,body,body_clean
0,2017-01-01 00:00:08+00:00,politics,GaryRuppert,<NA>,4,False,Kind of a shame for Bill Clinton that he's det...,Kind of a shame for Bill Clinton that he's det...
1,2017-01-01 00:00:48+00:00,politics,Kichigai,<NA>,7,False,"Please, McCain has done more to try and unite ...","Please, McCain has done more to try and unite ..."
2,2017-01-01 00:00:50+00:00,politics,GaryRuppert,<NA>,0,False,remember when Dems said Trump had no GOTV oper...,remember when Dems said Trump had no GOTV oper...
3,2017-01-01 00:00:54+00:00,AskTrumpSupporters,Rancel21342,supporter,3,False,Deport illegal immigrants and reduce the amoun...,Deport illegal immigrants and reduce the amoun...
4,2017-01-01 00:01:03+00:00,politics,SouffleStevens,<NA>,4,False,It really is surprising how many Donald Trump ...,It really is surprising how many Donald Trump ...


## 7. Output schema

**`submissions_clean`**: `id`, `created_utc`, `created_at`, `source_month`, `year`, `period` (T1_Y1 … T2_Y2), `term`, `subreddit`, `author`, `author_deleted`, `author_flair_text`, `stance`, `title`, `selftext`, `text` (= title + selftext), `is_self`, `is_removed`, `is_mod_post`, `domain`, `url`, `link_flair_text`, `score`, `num_comments`, `upvote_ratio`*, `removed_by_category`*, `distinguished`, `stickied`, `submission_mentions_trump`, `has_trump_comment`.

**`comments_clean`**: `id`, `submission_id`, `parent_id`, `is_top_level`, `created_utc`, `created_at`, `source_month`, `year`, `period`, `term`, `subreddit`, `author`, `author_deleted`, `author_flair_text`, `stance`, `body` (original), `body_clean` (quotes/URLs/HTML stripped, use this for NLP), `trump_only_in_quote` (True = Trump is mentioned only in quoted text, not by the author), `score`, `controversiality`, `is_submitter`*, `stickied`, `edited`.

\* available only for part of the period (mostly 2025–2026), because the Pushshift / Reddit API fields changed over time.

**Known limitations (for the report)**

- r/politics accounts for ~90% of comments. Analyses should be run per subreddit, or weighted, rather than pooled.
- Absolute volumes fall sharply from 2017 to 2026. Compare **shares** of Trump mentions, not raw counts.
- The keyword filter `\btrump\b` misses indirect references ("he", "the president", "45/47", "Drumpf").
- Removed/deleted content is not random: moderators remove rule-breaking and provocative posts more often, so the remaining sample is slightly biased.